# Pertemuan 5 — Classification & Regression
### Praktik: memprediksi respons kampanye dan potensi belanja pelanggan

<a href="https://colab.research.google.com/github/[username]/[nama-repo]/blob/main/pertemuan-05/05_Classification_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> Ganti `[username]` dan `[nama-repo]` pada tautan di atas (dan path `FILE_DATA` di bagian 2) dengan alamat repositori atau folder Google Drive kelas.

**Setelah menyelesaikan notebook ini, kamu bisa:**

1. Menjelaskan alur **supervised learning**: fitur (X), target (y), `fit`, dan `predict`.
2. Membedakan **klasifikasi** dan **regresi**.
3. Membuat **baseline** dan menjelaskan kenapa akurasi tinggi bisa menipu.
4. Melatih dan membandingkan Logistic Regression, Decision Tree, KNN, dan Random Forest.
5. Mengenali **overfitting** dari selisih skor data latih dan data uji.
6. Membaca model: koefisien, aturan pohon, dan peluang.
7. Melatih model regresi dan menilainya dengan MAE dan R².

**Dataset:** `marketing_segmen.csv`, hasil Pertemuan 4 (data bersih + fitur + segmen).

**Perkiraan waktu:** 70–80 menit.

---
## 1. Business case: siapa yang dikirimi penawaran, dan berapa nilainya?

Masih **Juli 2014** di *RetailKu* (perusahaan fiktif). Segmen pelanggan dari Pertemuan 4 sudah dipresentasikan, dan direksi senang. Sekarang manajer marketing datang dengan dua pertanyaan baru:

> *"Untuk kampanye ke-7, anggaran kita hanya cukup untuk mengirim penawaran ke **sebagian** pelanggan. Tolong beri tahu saya **siapa saja yang kemungkinan besar merespons**. Lalu satu lagi: setiap minggu ada pelanggan baru mendaftar. Bisakah kita **memperkirakan berapa nilai belanja** mereka, supaya tim CRM tahu siapa yang perlu disambut lebih serius?"*

Kali ini kita **punya jawaban di data**: kolom `Response` (merespons kampanye terakhir atau tidak) dan `TotalSpend` (total belanja dua tahun). Model belajar dari jawaban itu. Inilah **supervised learning**.

| | Klasifikasi | Regresi |
|---|---|---|
| Target berupa | **Kategori** (ya/tidak, A/B/C) | **Angka** kontinu |
| Pertanyaan RetailKu | Apakah pelanggan ini akan merespons? | Berapa total belanja pelanggan ini? |
| Target di data | `Response` (0/1) | `TotalSpend` |
| Contoh model | Logistic Regression, Decision Tree, KNN, Random Forest | Linear Regression, Decision Tree, Random Forest |
| Contoh ukuran | Akurasi, confusion matrix | MAE, R² |

> Pertemuan 6 dan 7 membahas cara **menilai** model secara mendalam. Hari ini fokusnya **melatih**, **membandingkan**, dan **membaca** model.

---
## 2. Menyiapkan alat dan data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

WARNA = "#2B6F7E"
WARNA_SOROT = "#B4501F"
WARNA_ABU = "#8A9BA8"

In [ ]:
# Hubungkan Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Tentukan path tempat Anda menyimpan file hasil Pertemuan 4 di Google Drive
FILE_DATA = '/content/drive/MyDrive/UNIPA 2026/Meet 5/marketing_segmen.csv'

df = pd.read_csv(FILE_DATA)
print("Ukuran:", df.shape)

In [ ]:
df[["ID", "Income", "Age", "Children", "TotalSpend", "TotalAcceptedPrev", "NamaSegmen", "Response"]].head()

---
## 3. Alur supervised learning

Semua model di scikit-learn dipakai dengan pola yang sama:

```python
model.fit(X_train, y_train)    # belajar dari contoh yang ada jawabannya
model.predict(X_test)          # menebak jawaban untuk data yang belum pernah dilihat
```

- **X** = fitur, yaitu informasi yang dipakai untuk menebak (pendapatan, umur, riwayat belanja, ...).
- **y** = target, yaitu jawaban yang ingin ditebak (`Response` atau `TotalSpend`).

Seperti di Pertemuan 3, data dibagi menjadi data latih dan data uji, dan semua langkah persiapan di-*fit* hanya pada data latih.

### 3.1 Kenali targetnya dulu

In [ ]:
jumlah = df["Response"].value_counts().sort_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.8))
ax1.bar(["Tidak merespons (0)", "Merespons (1)"], jumlah, color=[WARNA_ABU, WARNA_SOROT])
ax1.set_title("Target klasifikasi: Response", loc="left", fontweight="bold")
ax1.grid(axis="x", visible=False)
for i, n in enumerate(jumlah):
    ax1.text(i, n + 15, f"{n} ({n / len(df) * 100:.1f}%)", ha="center")

ax2.hist(df["TotalSpend"], bins=40, color=WARNA)
ax2.set_title("Target regresi: TotalSpend", loc="left", fontweight="bold")
ax2.set_xlabel("Total belanja 2 tahun")
plt.tight_layout()
plt.show()

💬 **Amati 1:** berapa persen pelanggan yang merespons? Jika kamu harus menebak tanpa model sama sekali, tebakan apa yang paling sering benar?

💬 **Amati 2:** `Response` hanya punya dua nilai, sedangkan `TotalSpend` bisa berapa saja. Inilah beda klasifikasi dan regresi.

---
## 4. Klasifikasi: siapa yang akan merespons?

### 4.1 Fitur, target, dan pembagian data

Kita memakai fitur dan pipeline persiapan yang **sama** dengan Pertemuan 3.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

URUTAN_PENDIDIKAN = ["Basic", "Graduation", "Master", "PhD"]

FITUR_ANGKA = ["Income", "Age", "Tenure_days", "Recency", "Children", "TotalSpend_log",
               "TotalPurchases", "AvgSpendPerPurchase", "NumDealsPurchases",
               "NumWebVisitsMonth", "TotalAcceptedPrev", "Complain"]
FITUR_KATEGORI = ["Education", "Marital_Status"]

def buat_persiapan(fitur_angka, scaling=True):
    """Pipeline persiapan dari Pertemuan 3. scaling=False untuk model berbasis pohon."""
    langkah_angka = [("isi_kosong", SimpleImputer(strategy="median"))]
    if scaling:
        langkah_angka.append(("skala", StandardScaler()))
    return ColumnTransformer([
        ("angka", Pipeline(langkah_angka), fitur_angka),
        ("ordinal", OrdinalEncoder(categories=[URUTAN_PENDIDIKAN],
                                   handle_unknown="use_encoded_value", unknown_value=-1), ["Education"]),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["Marital_Status"]),
    ], verbose_feature_names_out=False)

X = df[FITUR_ANGKA + FITUR_KATEGORI]
y = df["Response"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Data latih:", X_train.shape, "| Data uji:", X_test.shape)
print(f"Merespons di data uji: {y_test.sum()} dari {len(y_test)} pelanggan")

### 4.2 Baseline: model "malas"

Sebelum membuat model pintar, buat dulu **baseline**: model paling sederhana yang mungkin. Model yang kita buat nanti harus **lebih baik** daripada baseline; kalau tidak, untuk apa?

`DummyClassifier(strategy="most_frequent")` selalu menebak jawaban yang paling sering muncul.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

malas = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
tebakan_malas = malas.predict(X_test)

print("Tebakan unik model malas:", np.unique(tebakan_malas))
print(f"Akurasi model malas: {accuracy_score(y_test, tebakan_malas) * 100:.1f}%")

💬 **Amati:** akurasinya sekitar 85%, tanpa belajar apa pun! Tapi coba pikir: jika model ini dipakai untuk kampanye ke-7, berapa pelanggan yang akan dikirimi penawaran?

### 4.3 Confusion matrix: melihat ke dalam akurasi

**Akurasi** = persentase tebakan yang benar. Satu angka ini menyembunyikan banyak hal. **Confusion matrix** memecahnya menjadi empat kotak:

| | Ditebak tidak merespons | Ditebak merespons |
|---|---|---|
| **Sebenarnya tidak merespons** | ✅ Benar diabaikan | ❌ Penawaran terbuang |
| **Sebenarnya merespons** | ❌ **Responden terlewat** | ✅ **Responden tertangkap** |

In [ ]:
LABEL_KELAS = ["Tidak merespons", "Merespons"]

def gambar_confusion(y_benar, y_tebak, judul, ax):
    cm = confusion_matrix(y_benar, y_tebak, labels=[0, 1])
    warna = np.array([["#DCE9EC", "#F8E3D6"], ["#F8E3D6", "#DCE9EC"]])
    for i in range(2):
        for j in range(2):
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, color=warna[i, j]))
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=18, fontweight="bold")
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(1.5, -0.5)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(LABEL_KELAS)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(LABEL_KELAS)
    ax.set_xlabel("Tebakan model")
    ax.set_ylabel("Sebenarnya")
    ax.grid(False)
    ax.set_title(judul, loc="left", fontweight="bold")

def ringkas(nama, y_benar, y_tebak):
    cm = confusion_matrix(y_benar, y_tebak, labels=[0, 1])
    tertangkap, total_responden = cm[1, 1], cm[1].sum()
    return {
        "model": nama,
        "akurasi_%": round(accuracy_score(y_benar, y_tebak) * 100, 1),
        "responden_tertangkap": f"{tertangkap} dari {total_responden}",
        "%_tertangkap": float(round(tertangkap / total_responden * 100, 1)),
    }

fig, ax = plt.subplots(figsize=(5, 4))
gambar_confusion(y_test, tebakan_malas, "Model malas", ax)
plt.show()
ringkas("Model malas", y_test, tebakan_malas)

💬 **Diskusi:** model ini 85% "benar", tapi **tidak menangkap satu pun** pelanggan yang merespons. Untuk tim marketing, model ini tidak berguna sama sekali. Ingat pelajaran ini; Pertemuan 6 membahas ukuran yang lebih jujur.

### 4.4 Logistic Regression

Walaupun namanya "regression", **Logistic Regression** adalah model **klasifikasi**. Idenya:

1. Hitung skor dari penjumlahan fitur dikali bobot: `skor = b0 + b1·x1 + b2·x2 + ...`
2. Ubah skor menjadi **peluang** antara 0 dan 1 dengan kurva sigmoid.
3. Jika peluang ≥ 0,5, tebak "merespons".

In [ ]:
z = np.linspace(-6, 6, 200)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(z, 1 / (1 + np.exp(-z)), color=WARNA, linewidth=2.5)
ax.axhline(0.5, color=WARNA_SOROT, linestyle="--", linewidth=1)
ax.text(-6, 0.53, "batas 0,5", color=WARNA_SOROT)
ax.set_title("Kurva sigmoid: skor → peluang", loc="left", fontweight="bold")
ax.set_xlabel("Skor")
ax.set_ylabel("Peluang merespons")
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression

model_logreg = Pipeline([
    ("persiapan", buat_persiapan(FITUR_ANGKA)),
    ("model", LogisticRegression(max_iter=1000)),
])
model_logreg.fit(X_train, y_train)
tebakan_logreg = model_logreg.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
gambar_confusion(y_test, tebakan_malas, "Model malas", axes[0])
gambar_confusion(y_test, tebakan_logreg, "Logistic Regression", axes[1])
plt.tight_layout()
plt.show()

pd.DataFrame([ringkas("Model malas", y_test, tebakan_malas),
              ringkas("Logistic Regression", y_test, tebakan_logreg)])

Logistic Regression tidak hanya memberi tebakan 0/1, tetapi juga **peluang**:

In [ ]:
peluang = model_logreg.predict_proba(X_test)[:, 1]

contoh = X_test[["Income", "Recency", "TotalAcceptedPrev"]].copy()
contoh["peluang_merespons"] = peluang.round(2)
contoh["tebakan"] = tebakan_logreg
contoh["sebenarnya"] = y_test.values
contoh.sort_values("peluang_merespons", ascending=False).head(8)

💬 **Amati:** pelanggan dengan peluang tertinggi, apa kesamaan mereka? Lihat `TotalAcceptedPrev` dan `Recency`.

### 4.5 Decision Tree: model berupa aturan "jika ... maka ..."

**Decision Tree** memecah data dengan pertanyaan ya/tidak, seperti permainan tebak-tebakan. Keunggulannya: aturannya bisa dibaca manusia.

Pohon membandingkan nilai dengan batas (misalnya `Recency <= 20`), bukan menghitung jarak. Jadi pohon **tidak memerlukan scaling**, dan tanpa scaling batasnya lebih mudah dibaca.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

model_pohon = Pipeline([
    ("persiapan", buat_persiapan(FITUR_ANGKA, scaling=False)),
    ("model", DecisionTreeClassifier(max_depth=3, random_state=42)),
])
model_pohon.fit(X_train, y_train)
tebakan_pohon = model_pohon.predict(X_test)

fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(model_pohon.named_steps["model"],
          feature_names=model_pohon.named_steps["persiapan"].get_feature_names_out(),
          class_names=LABEL_KELAS, filled=True, rounded=True, fontsize=9, impurity=False, ax=ax)
plt.show()

ringkas("Decision Tree (depth 3)", y_test, tebakan_pohon)

💬 **Amati:**

1. Pertanyaan pertama (paling atas) memakai fitur apa? Kenapa menurutmu fitur itu dipilih lebih dulu?
2. Ikuti satu jalur sampai kotak berwarna oranye ("Merespons"). Tuliskan aturannya dalam satu kalimat untuk tim marketing.

### 4.6 K-Nearest Neighbors (KNN): tanya tetangga terdekat

Ingat jarak di Pertemuan 4? **KNN** memakai ide yang sama: untuk pelanggan baru, cari **K pelanggan paling mirip** di data latih, lalu ikuti suara terbanyak. Karena memakai jarak, KNN **wajib scaling**.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model_knn = Pipeline([
    ("persiapan", buat_persiapan(FITUR_ANGKA)),
    ("model", KNeighborsClassifier(n_neighbors=5)),
])
model_knn.fit(X_train, y_train)
tebakan_knn = model_knn.predict(X_test)
ringkas("KNN (k=5)", y_test, tebakan_knn)

### 4.7 Random Forest: banyak pohon, satu suara

Satu pohon mudah "salah paham". **Random Forest** melatih ratusan pohon, masing-masing dengan sampel data dan fitur yang sedikit berbeda, lalu mengambil **suara terbanyak**. Hasilnya biasanya lebih stabil, tetapi lebih sulit dibaca.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_hutan = Pipeline([
    ("persiapan", buat_persiapan(FITUR_ANGKA, scaling=False)),
    ("model", RandomForestClassifier(n_estimators=300, random_state=42)),
])
model_hutan.fit(X_train, y_train)
tebakan_hutan = model_hutan.predict(X_test)
ringkas("Random Forest", y_test, tebakan_hutan)

### 4.8 Membandingkan semua model

In [ ]:
semua_model = {
    "Model malas": malas,
    "Logistic Regression": model_logreg,
    "Decision Tree (depth 3)": model_pohon,
    "KNN (k=5)": model_knn,
    "Random Forest": model_hutan,
}

hasil = []
for nama, model in semua_model.items():
    baris = ringkas(nama, y_test, model.predict(X_test))
    baris["akurasi_latih_%"] = round(model.score(X_train, y_train) * 100, 1)
    hasil.append(baris)

tabel_klasifikasi = pd.DataFrame(hasil).set_index("model")
tabel_klasifikasi

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
posisi = np.arange(len(tabel_klasifikasi))
ax.bar(posisi - 0.2, tabel_klasifikasi["akurasi_%"], width=0.4, color=WARNA, label="Akurasi (%)")
ax.bar(posisi + 0.2, tabel_klasifikasi["%_tertangkap"], width=0.4, color=WARNA_SOROT, label="% responden tertangkap")
ax.set_xticks(posisi)
ax.set_xticklabels(tabel_klasifikasi.index, rotation=10)
ax.set_ylim(0, 115)
ax.set_title("Akurasi vs responden yang tertangkap (data uji)", loc="left", fontweight="bold")
ax.legend(loc="upper center", ncol=2, frameon=False)
ax.grid(axis="x", visible=False)
plt.show()

💬 **Diskusi:**

1. Perbedaan akurasi antarmodel hanya beberapa persen. Bagaimana dengan perbedaan "% responden tertangkap"?
2. Bandingkan `akurasi_latih_%` dengan `akurasi_%` (uji) untuk Random Forest. Apa yang terjadi?
3. Jika kamu manajer marketing, ukuran mana yang lebih penting: akurasi atau responden tertangkap? Kenapa?

---
## 5. Overfitting: menghafal vs memahami

Model yang **terlalu rumit** bisa menghafal data latih, termasuk kebetulan dan noise, tetapi gagal pada data baru. Seperti mahasiswa yang menghafal kunci jawaban latihan, lalu bingung saat soal ujian sedikit berbeda.

Kita lihat dengan mengubah kedalaman pohon (`max_depth`).

In [ ]:
daftar_kedalaman = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20]
skor_latih, skor_uji = [], []

for d in daftar_kedalaman:
    m = Pipeline([
        ("persiapan", buat_persiapan(FITUR_ANGKA, scaling=False)),
        ("model", DecisionTreeClassifier(max_depth=d, random_state=42)),
    ]).fit(X_train, y_train)
    skor_latih.append(m.score(X_train, y_train) * 100)
    skor_uji.append(m.score(X_test, y_test) * 100)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(daftar_kedalaman, skor_latih, marker="o", color=WARNA, label="Data latih")
ax.plot(daftar_kedalaman, skor_uji, marker="o", color=WARNA_SOROT, label="Data uji")
ax.set_title("Akurasi Decision Tree menurut kedalaman", loc="left", fontweight="bold")
ax.set_xlabel("max_depth (kerumitan model)")
ax.set_ylabel("Akurasi (%)")
ax.legend()
plt.show()

💬 **Amati:**

1. Apa yang terjadi pada akurasi data latih saat pohon makin dalam? Dan pada data uji?
2. Di kedalaman berapa selisih keduanya mulai melebar?
3. Kenapa kita **tidak boleh** memilih model hanya berdasarkan skor data latih?

| Kondisi | Data latih | Data uji | Artinya |
|---|---|---|---|
| **Underfitting** | Rendah | Rendah | Model terlalu sederhana |
| **Pas** | Baik | Baik, selisih kecil | Model menangkap pola umum |
| **Overfitting** | Sangat tinggi | Jauh lebih rendah | Model menghafal |

> Memilih `max_depth` dengan melihat data uji berulang-ulang juga berisiko, karena lama-lama data uji ikut "dihafal". Cara yang benar (*cross-validation*) dibahas di Pertemuan 7.

---
## 6. Membaca dan memakai model klasifikasi

### 6.1 Fitur apa yang paling berpengaruh?

Pada Logistic Regression, setiap fitur punya **koefisien**. Karena fitur angka sudah di-scaling, koefisiennya bisa dibandingkan: positif menaikkan peluang merespons, negatif menurunkannya.

In [ ]:
nama_fitur = model_logreg.named_steps["persiapan"].get_feature_names_out()
koef = pd.Series(model_logreg.named_steps["model"].coef_[0], index=nama_fitur).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(koef.index, koef.values, color=[WARNA_SOROT if v > 0 else WARNA for v in koef.values])
ax.axvline(0, color="#1B2A35", linewidth=0.8)
ax.set_title("Koefisien Logistic Regression", loc="left", fontweight="bold")
ax.set_xlabel("← menurunkan peluang | menaikkan peluang →")
ax.grid(axis="y", visible=False)
plt.show()

💬 **Diskusi:** tiga fitur apa yang paling menaikkan peluang merespons? Masuk akalkah bagi bisnis? Ingat: ini **hubungan**, belum tentu sebab-akibat.

### 6.2 Daftar prioritas kampanye

Anggaran hanya cukup untuk sebagian pelanggan. Daripada memakai tebakan 0/1, urutkan pelanggan berdasarkan **peluang merespons**, lalu kirim penawaran ke yang teratas.

In [ ]:
urutan = np.argsort(-peluang)                      # dari peluang tertinggi
n_kirim = int(len(urutan) * 0.20)                  # anggaran: 20% pelanggan
tertangkap_model = y_test.values[urutan[:n_kirim]].sum()
total_responden = y_test.sum()

# Jika dipilih acak, rata-rata kita menangkap 20% dari semua responden
tertangkap_acak = round(total_responden * 0.20)

print(f"Penawaran dikirim ke {n_kirim} pelanggan (20% data uji)")
print(f"Dipilih acak       : ±{tertangkap_acak} dari {total_responden} responden tertangkap")
print(f"Dipilih oleh model : {tertangkap_model} dari {total_responden} responden tertangkap "
      f"({tertangkap_model / total_responden * 100:.0f}%)")

fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(["Pilih acak", "Pilih dengan model"], [tertangkap_acak, tertangkap_model], color=[WARNA_ABU, WARNA_SOROT])
ax.set_title(f"Responden tertangkap dengan anggaran {n_kirim} penawaran", loc="left", fontweight="bold")
ax.set_xlabel(f"Jumlah responden (dari {total_responden})")
ax.grid(axis="y", visible=False)
plt.show()

💬 **Diskusi:** dengan jumlah penawaran yang sama, berapa kali lebih banyak responden yang tertangkap jika memakai model? Berapa batas peluang yang sebaiknya dipakai, dan bagaimana jika biaya per penawaran naik? Itu bahan Pertemuan 7.

---
## 7. Regresi: berapa potensi belanja pelanggan?

### 7.1 Fitur apa yang boleh dipakai?

Tim CRM ingin memperkirakan nilai belanja **pelanggan yang baru mendaftar**. Saat mendaftar, yang kita ketahui hanyalah **profilnya**. Riwayat belanja belum ada.

In [ ]:
FITUR_PROFIL = ["Income", "Age", "Children", "Tenure_days"]
TARGET_REG = "TotalSpend"

X_reg = df[FITUR_PROFIL + FITUR_KATEGORI]
y_reg = df[TARGET_REG]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
print("Data latih:", Xr_train.shape, "| Data uji:", Xr_test.shape)

💬 **Detektif kebocoran:** kenapa fitur berikut **tidak boleh** dipakai untuk memprediksi `TotalSpend`?

| Fitur | Masalah |
|---|---|
| `MntWines`, `MntMeatProducts`, ... | |
| `TotalSpend_log`, `AvgSpendPerPurchase` | |
| `Segmen` dari Pertemuan 4 | |

<details>
<summary><b>Penjelasan</b></summary>

Semuanya **dihitung dari target itu sendiri**. `TotalSpend` adalah jumlah kolom `Mnt...`; `TotalSpend_log` dan `AvgSpendPerPurchase` diturunkan dari `TotalSpend`; dan segmen Pertemuan 4 dibentuk memakai `TotalSpend_log`. Model akan terlihat hebat, padahal hanya "menyalin" jawaban. Ini kebocoran bentuk 1 dari Pertemuan 3. Selain itu, pelanggan baru belum punya data belanja.

</details>

### 7.2 Baseline dan ukuran regresi

Baseline regresi: **selalu menebak rata-rata** `TotalSpend` data latih.

| Ukuran | Arti | Semakin baik jika |
|---|---|---|
| **MAE** (*Mean Absolute Error*) | Rata-rata selisih tebakan dan nilai sebenarnya, dalam satuan target | Kecil |
| **R²** | Seberapa jauh model lebih baik daripada sekadar menebak rata-rata (0 = sama saja, 1 = sempurna) | Mendekati 1 |

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score

def ringkas_reg(nama, model):
    tebak = model.predict(Xr_test)
    return {
        "model": nama,
        "MAE": round(mean_absolute_error(yr_test, tebak)),
        "R2_uji": round(r2_score(yr_test, tebak), 3),
        "R2_latih": round(model.score(Xr_train, yr_train), 3),
    }

reg_malas = DummyRegressor(strategy="mean").fit(Xr_train, yr_train)
print(f"Tebakan model malas untuk semua pelanggan: {reg_malas.predict(Xr_test[:1])[0]:,.0f}")
ringkas_reg("Model malas (rata-rata)", reg_malas)

### 7.3 Linear Regression

**Linear Regression** mencari garis (atau bidang) terbaik: `TotalSpend ≈ b0 + b1·Income + b2·Age + b3·Children + ...`. Setiap koefisien menunjukkan perubahan tebakan jika satu fitur naik satu satuan, dengan fitur lain tetap.

In [ ]:
from sklearn.linear_model import LinearRegression

reg_linear = Pipeline([
    ("persiapan", buat_persiapan(FITUR_PROFIL)),
    ("model", LinearRegression()),
]).fit(Xr_train, yr_train)

ringkas_reg("Linear Regression", reg_linear)

In [ ]:
# Mengembalikan koefisien ke satuan asli (karena fitur angka sudah di-scaling)
sd = reg_linear.named_steps["persiapan"].named_transformers_["angka"].named_steps["skala"].scale_
koef_asli = reg_linear.named_steps["model"].coef_[:len(FITUR_PROFIL)] / sd

pd.DataFrame({
    "perubahan": ["+10.000 pendapatan", "+1 tahun umur", "+1 anak", "+1 tahun berlangganan"],
    "perubahan_TotalSpend": (koef_asli * [10_000, 1, 1, 365]).round(0),
}, index=FITUR_PROFIL)

💬 **Amati:** setiap tambahan satu anak mengubah perkiraan belanja sebesar berapa? Cocokkah dengan segmen "Premium" dan "Hemat" di Pertemuan 4?

### 7.4 Decision Tree dan Random Forest untuk regresi

Model pohon juga bisa menebak angka: setiap daun berisi **rata-rata** target pelanggan yang jatuh ke daun itu.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

reg_pohon = Pipeline([
    ("persiapan", buat_persiapan(FITUR_PROFIL, scaling=False)),
    ("model", DecisionTreeRegressor(max_depth=4, random_state=42)),
]).fit(Xr_train, yr_train)

reg_hutan = Pipeline([
    ("persiapan", buat_persiapan(FITUR_PROFIL, scaling=False)),
    ("model", RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=42)),
]).fit(Xr_train, yr_train)

tabel_regresi = pd.DataFrame([
    ringkas_reg("Model malas (rata-rata)", reg_malas),
    ringkas_reg("Linear Regression", reg_linear),
    ringkas_reg("Decision Tree (depth 4)", reg_pohon),
    ringkas_reg("Random Forest", reg_hutan),
]).set_index("model")
tabel_regresi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
batas = [0, yr_test.max() * 1.05]
for ax, (nama, model, warna) in zip(axes, [("Linear Regression", reg_linear, WARNA),
                                           ("Random Forest", reg_hutan, WARNA_SOROT)]):
    ax.scatter(yr_test, model.predict(Xr_test), s=10, alpha=0.5, color=warna)
    ax.plot(batas, batas, color="#1B2A35", linestyle="--", linewidth=1)
    ax.set_title(nama, loc="left", fontweight="bold")
    ax.set_xlabel("TotalSpend sebenarnya")
axes[0].set_ylabel("TotalSpend tebakan")
plt.tight_layout()
plt.show()

💬 **Amati:**

1. Titik yang tepat di garis putus-putus berarti tebakan sempurna. Di bagian mana tebakan paling meleset?
2. Linear Regression kadang menebak belanja **negatif**. Masuk akalkah? Kenapa itu bisa terjadi?
3. MAE Random Forest sekitar berapa? Dibandingkan rata-rata belanja pelanggan (sekitar 600), apakah meleset sebesar itu bisa diterima untuk keperluan tim CRM?

---
## 8. Memakai model untuk pelanggan baru

Tiga pelanggan baru mendaftar minggu ini. Berapa perkiraan potensi belanja mereka selama dua tahun?

In [ ]:
pelanggan_baru = pd.DataFrame([
    {"Nama": "Bu Rina",  "Income": 82000,  "Age": 52, "Children": 0, "Tenure_days": 7,
     "Education": "Master", "Marital_Status": "Married"},
    {"Nama": "Pak Dodi", "Income": 29000,  "Age": 33, "Children": 1, "Tenure_days": 5,
     "Education": "Graduation", "Marital_Status": "Together"},
    {"Nama": "Mbak Sari", "Income": np.nan, "Age": 41, "Children": 3, "Tenure_days": 2,
     "Education": "PhD", "Marital_Status": "Single"},
])

fitur_baru = pelanggan_baru[FITUR_PROFIL + FITUR_KATEGORI]
pelanggan_baru["perkiraan_linear"] = reg_linear.predict(fitur_baru).round(0)
pelanggan_baru["perkiraan_hutan"] = reg_hutan.predict(fitur_baru).round(0)
pelanggan_baru[["Nama", "Income", "Children", "perkiraan_linear", "perkiraan_hutan"]]

💬 **Amati:**

1. Siapa yang sebaiknya disambut lebih serius oleh tim CRM?
2. Pendapatan Mbak Sari kosong. Dengan nilai apa pipeline mengisinya, dan bagaimana itu memengaruhi perkiraan?
3. Pelanggan baru belum punya riwayat kampanye, jadi model klasifikasi di bagian 4 belum bisa dipakai untuk mereka. Fitur apa yang membuat model itu bergantung pada riwayat?

---
## 9. Menyimpan hasil

Kita simpan prediksi pada data uji. Pertemuan 6 akan memakainya untuk menilai model dengan lebih teliti.

In [ ]:
prediksi_uji = pd.DataFrame({
    "ID": df.loc[X_test.index, "ID"],
    "Response": y_test,
    "peluang_logreg": model_logreg.predict_proba(X_test)[:, 1].round(4),
    "peluang_hutan": model_hutan.predict_proba(X_test)[:, 1].round(4),
    "tebakan_logreg": tebakan_logreg,
    "tebakan_pohon": tebakan_pohon,
    "tebakan_hutan": tebakan_hutan,
})

FILE_PREDIKSI = "prediksi_response_uji.csv"
prediksi_uji.to_csv(FILE_PREDIKSI, index=False)
print("Tersimpan:", FILE_PREDIKSI, "| ukuran:", prediksi_uji.shape)

In [ ]:
# OPSIONAL: unduh file ke komputer (hapus tanda # di dua baris berikut)

# from google.colab import files
# files.download(FILE_PREDIKSI)

---
## 10. Latihan mandiri

**Latihan 1 — Segmen sebagai fitur.** Tambahkan `NamaSegmen` dari Pertemuan 4 ke fitur klasifikasi (one-hot encoding), latih ulang Logistic Regression, dan bandingkan hasil `ringkas()`. Apakah segmen membantu?
*Petunjuk:* tambahkan jalur `("segmen", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["NamaSegmen"])` di `ColumnTransformer`.

In [ ]:
# Tulis jawaban Latihan 1 di sini

**Latihan 2 — Berapa tetangga?** Latih KNN dengan `n_neighbors` = 1, 3, 5, 11, 21, 51. Catat akurasi latih dan uji. Nilai K berapa yang overfitting, dan berapa yang underfitting?

In [ ]:
# Tulis jawaban Latihan 2 di sini

**Latihan 3 — Fitur riwayat untuk regresi.** Tambahkan `TotalPurchases` dan `NumWebVisitsMonth` ke `FITUR_PROFIL`, lalu latih ulang Linear Regression dan Random Forest. Berapa kenaikan R²? Apakah fitur itu **tersedia** untuk pelanggan yang baru mendaftar?

In [ ]:
# Tulis jawaban Latihan 3 di sini

**Latihan 4 — Kedalaman pohon regresi.** Buat grafik R² latih dan uji untuk `DecisionTreeRegressor` dengan `max_depth` 1 sampai 15, seperti bagian 5. Di mana overfitting mulai terjadi?

In [ ]:
# Tulis jawaban Latihan 4 di sini

**Latihan 5 (tantangan) — Target log.** `TotalSpend` sangat miring. Latih Linear Regression untuk menebak `np.log1p(TotalSpend)`, lalu kembalikan tebakan dengan `np.expm1()` sebelum menghitung MAE. Apakah MAE membaik? Apakah tebakan negatif masih muncul?

In [ ]:
# Tulis jawaban Latihan 5 di sini

---
## 11. Tugas: rekomendasi model untuk manajer marketing

Lengkapi tabel berikut dari hasil notebook-mu.

| Pertanyaan bisnis | Jenis tugas | Model terbaik menurutmu | Ukuran yang kamu pakai | Hasil |
|---|---|---|---|---|
| Siapa yang akan merespons kampanye ke-7? | Klasifikasi | | | |
| Berapa potensi belanja pelanggan baru? | Regresi | | | |

Lalu tulis **memo 4–6 kalimat** untuk manajer marketing: model mana yang kamu rekomendasikan, kenapa akurasi saja tidak cukup untuk menilai model klasifikasi, dan berapa responden yang bisa tertangkap jika penawaran dikirim ke 20% pelanggan teratas.

---
## Rangkuman

| Konsep | Inti | Alat |
|---|---|---|
| Supervised learning | Belajar dari contoh yang ada jawabannya | `fit`, `predict` |
| Klasifikasi vs regresi | Target kategori vs angka | — |
| Baseline | Model malas sebagai batas bawah | `DummyClassifier`, `DummyRegressor` |
| Akurasi bisa menipu | Kelas tidak seimbang; lihat confusion matrix | `confusion_matrix` |
| Model klasifikasi | Garis + sigmoid, aturan pohon, tetangga, banyak pohon | `LogisticRegression`, `DecisionTreeClassifier`, `KNeighborsClassifier`, `RandomForestClassifier` |
| Overfitting | Latih tinggi, uji rendah | `max_depth`, bandingkan skor latih vs uji |
| Peluang | Mengurutkan prioritas, bukan hanya 0/1 | `predict_proba` |
| Model regresi | Garis, pohon, banyak pohon | `LinearRegression`, `DecisionTreeRegressor`, `RandomForestRegressor` |
| Ukuran regresi | Rata-rata meleset dan perbandingan dengan rata-rata | MAE, R² |

**Pertemuan berikutnya, Model Evaluation:** akurasi saja menipu. Kita mengenal precision, recall, F1, ROC-AUC untuk klasifikasi, serta MAE, RMSE, dan R² untuk regresi secara lebih mendalam, memakai file `prediksi_response_uji.csv`.